In [1]:
import torch
from mmcv_ops.nms import nms, soft_nms, nms_match

/home/zzhu622/.conda/envs/mmcv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# generate bboxes randomly (N, 4)
bboxes = torch.randn((100, 4), dtype=torch.float32).cuda()
# prevent negative elements and wrong bboxes
for i in range(100):
    bbox = bboxes[i, :]
    while ((bbox[0] > bbox[2]) or (bbox[1] > bbox[3]) or torch.any(bbox < 0)):
        bbox = torch.randn((4,), dtype=torch.float32).cuda()
    bboxes[i, :] = bbox
# generate scores randomly (N)
scores = torch.randn((100), dtype=torch.float32).cuda()
# create iou_threshold
iou_threshold = 0.5
# create offset
offset = 0
# create score_threshold
score_threshold = 0.5
# create max_num
max_num = 10

In [ ]:
# print feats
print('bboxes:\n')
print(bboxes)
print('scores:\n')
print(scores)

bboxes:

tensor([[1.5548e-02, 8.4136e-01, 2.2140e+00, 1.1227e+00],
        [3.9589e-01, 4.2712e-01, 8.3111e-01, 1.2112e+00],
        [9.5973e-01, 8.7726e-02, 1.1555e+00, 1.9982e-01],
        [4.7430e-01, 5.9334e-01, 7.4834e-01, 1.4182e+00],
        [7.9888e-02, 3.1950e-01, 9.3099e-01, 4.2168e-01],
        [3.7546e-01, 9.5104e-01, 8.4887e-01, 1.0182e+00],
        [4.1854e-01, 6.8656e-01, 1.8186e+00, 1.1965e+00],
        [6.1103e-02, 3.9341e-01, 1.1632e+00, 5.2295e-01],
        [1.0185e+00, 8.3742e-02, 2.0150e+00, 1.3492e+00],
        [4.7370e-01, 2.9453e-01, 6.4429e-01, 1.0724e+00],
        [2.5134e-01, 8.0686e-01, 1.4189e+00, 1.4090e+00],
        [7.4847e-01, 1.7694e-01, 9.7650e-01, 5.8352e-01],
        [2.2688e-01, 9.9430e-02, 2.6150e-01, 1.3113e+00],
        [4.1626e-01, 8.5555e-01, 4.6824e-01, 8.6780e-01],
        [3.2835e-01, 4.2112e-01, 1.1033e+00, 7.8918e-01],
        [5.0976e-02, 1.1563e+00, 1.4852e+00, 2.1365e+00],
        [1.0668e-01, 4.0092e-02, 1.2094e-01, 1.5863e+00],
     

### 1. `forward` Function in NMSop
The following is the forward function in NMSop Module.
```python
    @staticmethod
    def forward(ctx: Any, bboxes: Tensor, scores: Tensor, iou_threshold: float,
                offset: int, score_threshold: float, max_num: int) -> Tensor:
        is_filtering_by_score = score_threshold > 0
        device = bboxes.device
        if is_filtering_by_score:
            valid_mask = scores > score_threshold
            bboxes, scores = bboxes[valid_mask], scores[valid_mask]
            valid_inds = torch.nonzero(valid_mask,
                                       as_tuple=False).squeeze(dim=1)

        if device == torch.device('cpu'):
            nms_forward = ext_module.nms_forward_cpu
        else:
            nms_forward = ext_module.nms_forward_cuda

        inds = nms_forward(bboxes,
                            scores,
                            iou_threshold=float(iou_threshold),
                            offset=offset)

        if max_num > 0:
            inds = inds[:max_num]
        if is_filtering_by_score:
            inds = valid_inds[inds]
        return inds
```

### 3. `nms_forward_cpu`
Let's see `nms_forward_cpu` firstly.

```cpp
Tensor nms_forward_cpu(Tensor boxes, Tensor scores, float iou_threshold, int offset) {
  if (boxes.numel() == 0) {
    return at::empty({0}, boxes.options().dtype(at::kLong));
  }
  // equals to bboxes[:, 0] bboxes[:, 1] bboxes[:, 2] bboxes[:, 3]
  auto x1_t = boxes.select(1, 0).contiguous();
  auto y1_t = boxes.select(1, 1).contiguous();
  auto x2_t = boxes.select(1, 2).contiguous();
  auto y2_t = boxes.select(1, 3).contiguous();
  // calcute the area
  // using offset to avoid negative areas?
  // calculate the area of each box
  Tensor areas_t = (x2_t - x1_t + offset) * (y2_t - y1_t + offset);
  // get the index
  // note that the return value of scores.sort is a tuple (tensor, index)
  auto order_t = std::get<1>(scores.sort(0, /* descending=*/true));

  auto nboxes = boxes.size(0);
  // create mask vector
  Tensor select_t = at::ones({nboxes}, boxes.options().dtype(at::kBool));
  // get data pointer
  auto select = select_t.data_ptr<bool>();
  auto order = order_t.data_ptr<int64_t>();
  auto x1 = x1_t.data_ptr<float>();
  auto y1 = y1_t.data_ptr<float>();
  auto x2 = x2_t.data_ptr<float>();
  auto y2 = y2_t.data_ptr<float>();
  auto areas = areas_t.data_ptr<float>();
  // loop
  for (int64_t _i = 0; _i < nboxes; _i++) {
    if (select[_i] == false) continue;
    auto i = order[_i];
    auto ix1 = x1[i];
    auto iy1 = y1[i];
    auto ix2 = x2[i];
    auto iy2 = y2[i];
    auto iarea = areas[i];

    for (int64_t _j = _i + 1; _j < nboxes; _j++) {
      if (select[_j] == false) continue;
      auto j = order[_j];
      // calculate the IoU
      auto xx1 = std::max(ix1, x1[j]);
      auto yy1 = std::max(iy1, y1[j]);
      auto xx2 = std::min(ix2, x2[j]);
      auto yy2 = std::min(iy2, y2[j]);

      auto w = std::max(0.f, xx2 - xx1 + offset);
      auto h = std::max(0.f, yy2 - yy1 + offset);
      // calculate the intersection area
      auto inter = w * h;
      // calculate the IoU
      auto ovr = inter / (iarea + areas[j] - inter);
      if (ovr > iou_threshold) select[_j] = false;
    }
  }
  return order_t.masked_select(select_t);
}
```cpp

In [4]:
valid_bboxes, valid_inds = nms(bboxes, scores, iou_threshold, offset, score_threshold, max_num)

In [5]:
print("valid_bboxes:\n")
print(valid_bboxes)
print("valid_inds:\n")
print(valid_inds)

valid_bboxes:

tensor([[0.3284, 0.4211, 1.1033, 0.7892, 2.8267],
        [0.3801, 0.3439, 0.5653, 2.1761, 2.0595],
        [0.0590, 0.2482, 0.5444, 0.4267, 1.8478],
        [0.2210, 0.2869, 1.1955, 0.9615, 1.7301],
        [0.3755, 0.9510, 0.8489, 1.0182, 1.5835],
        [0.2922, 0.8320, 0.5758, 0.9646, 1.5504],
        [0.2137, 0.3733, 0.3048, 0.5690, 1.5210],
        [0.9596, 0.3748, 1.6446, 1.6628, 1.5041],
        [0.3775, 0.6516, 1.4118, 1.8723, 1.4651],
        [0.4739, 0.5276, 1.8598, 0.8488, 1.3831]], device='cuda:0')
valid_inds:

tensor([14, 48, 79, 42,  5, 44, 78, 61, 34, 68], device='cuda:0')


### 3. `softnms_forward_cpu`

```cpp
Tensor softnms_forward_cpu(Tensor boxes, Tensor scores, Tensor dets,
                   float iou_threshold, float sigma, float min_score,
                   int method, int offset) {
  if (boxes.numel() == 0) {
    return at::empty({0}, boxes.options().dtype(at::kLong));
  }

  auto x1_t = boxes.select(1, 0).contiguous();
  auto y1_t = boxes.select(1, 1).contiguous();
  auto x2_t = boxes.select(1, 2).contiguous();
  auto y2_t = boxes.select(1, 3).contiguous();
  auto scores_t = scores.clone();

  Tensor areas_t = (x2_t - x1_t + offset) * (y2_t - y1_t + offset);

  auto nboxes = boxes.size(0);
  auto x1 = x1_t.data_ptr<float>();
  auto y1 = y1_t.data_ptr<float>();
  auto x2 = x2_t.data_ptr<float>();
  auto y2 = y2_t.data_ptr<float>();
  auto sc = scores_t.data_ptr<float>();
  auto areas = areas_t.data_ptr<float>();
  auto de = dets.data_ptr<float>();

  int64_t pos = 0;
  Tensor inds_t = at::arange(nboxes, boxes.options().dtype(at::kLong));
  auto inds = inds_t.data_ptr<int64_t>();

  for (int64_t i = 0; i < nboxes; i++) {
    auto max_score = sc[i];
    auto max_pos = i;

    pos = i + 1;
    // get max box
    while (pos < nboxes) {
      if (max_score < sc[pos]) {
        max_score = sc[pos];
        max_pos = pos;
      }
      pos = pos + 1;
    }
    // swap
    auto ix1 = de[i * 5 + 0] = x1[max_pos];
    auto iy1 = de[i * 5 + 1] = y1[max_pos];
    auto ix2 = de[i * 5 + 2] = x2[max_pos];
    auto iy2 = de[i * 5 + 3] = y2[max_pos];
    auto iscore = de[i * 5 + 4] = sc[max_pos];
    auto iarea = areas[max_pos];
    auto iind = inds[max_pos];
    x1[max_pos] = x1[i];
    y1[max_pos] = y1[i];
    x2[max_pos] = x2[i];
    y2[max_pos] = y2[i];
    sc[max_pos] = sc[i];
    areas[max_pos] = areas[i];
    inds[max_pos] = inds[i];
    x1[i] = ix1;
    y1[i] = iy1;
    x2[i] = ix2;
    y2[i] = iy2;
    sc[i] = iscore;
    areas[i] = iarea;
    inds[i] = iind;

    pos = i + 1;
    while (pos < nboxes) {
      auto xx1 = std::max(ix1, x1[pos]);
      auto yy1 = std::max(iy1, y1[pos]);
      auto xx2 = std::min(ix2, x2[pos]);
      auto yy2 = std::min(iy2, y2[pos]);

      auto w = std::max(0.f, xx2 - xx1 + offset);
      auto h = std::max(0.f, yy2 - yy1 + offset);
      auto inter = w * h;
      auto ovr = inter / (iarea + areas[pos] - inter);

      float weight = 1.;
      if (method == 0) {
        if (ovr >= iou_threshold) weight = 0;
      } else if (method == 1) {
        if (ovr >= iou_threshold) weight = 1 - ovr;
      } else if (method == 2) {
        weight = std::exp(-(ovr * ovr) / sigma);
      }
      sc[pos] *= weight;
      // if box score falls below threshold, discard the box by
      // swapping with last box update N
      if (sc[pos] < min_score) {
        x1[pos] = x1[nboxes - 1];
        y1[pos] = y1[nboxes - 1];
        x2[pos] = x2[nboxes - 1];
        y2[pos] = y2[nboxes - 1];
        sc[pos] = sc[nboxes - 1];
        areas[pos] = areas[nboxes - 1];
        inds[pos] = inds[nboxes - 1];
        nboxes = nboxes - 1;
        pos = pos - 1;
      }
      pos = pos + 1;
    }
  }
  return inds_t.slice(0, 0, nboxes);
}
```cpp

In [ ]:
valid_bboxes, valid_inds = soft_nms(bboxes, scores, iou_threshold)

In [7]:
print("valid_bboxes:\n")
print(valid_bboxes)
print("valid_inds:\n")
print(valid_inds)

valid_bboxes:

tensor([[0.3284, 0.4211, 1.1033, 0.7892, 2.8267],
        [0.3801, 0.3439, 0.5653, 2.1761, 2.0595],
        [0.0590, 0.2482, 0.5444, 0.4267, 1.8478],
        [0.2210, 0.2869, 1.1955, 0.9615, 1.7301],
        [0.3755, 0.9510, 0.8489, 1.0182, 1.5835],
        [0.2922, 0.8320, 0.5758, 0.9646, 1.5504],
        [0.2137, 0.3733, 0.3048, 0.5690, 1.5210],
        [0.9596, 0.3748, 1.6446, 1.6628, 1.5041],
        [0.3775, 0.6516, 1.4118, 1.8723, 1.4651],
        [0.4739, 0.5276, 1.8598, 0.8488, 1.3831],
        [0.2787, 0.1046, 0.3161, 1.0922, 1.2813],
        [0.2416, 0.2960, 1.7136, 0.3231, 1.2508],
        [0.2376, 0.8057, 0.3657, 1.2862, 1.2079],
        [1.1569, 0.1654, 2.5206, 0.3923, 1.1581],
        [0.4399, 0.1438, 1.1821, 0.4016, 1.1317],
        [0.4163, 0.8556, 0.4682, 0.8678, 1.1077],
        [0.1067, 0.0401, 0.1209, 1.5863, 1.0708],
        [0.9597, 0.0877, 1.1555, 0.1998, 0.9765],
        [0.4185, 0.6866, 1.8186, 1.1965, 0.9756],
        [0.2543, 0.5031, 1.1477, 0.